In [1]:
%pip -q install torchao==0.16.0
%pip install -q "transformers==4.57.1" "peft==0.17.1" "accelerate==1.11.0"

from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
from pathlib import Path
from importlib.metadata import version
from decimal import Decimal
import gc
import hashlib
import json
import re
import unicodedata
from datasets import load_dataset
from PIL import ImageDraw

import pandas as pd
import torch
import transformers
from peft import PeftModel
from PIL import Image
from transformers import AutoProcessor, GenerationConfig, Qwen3VLForConditionalGeneration, set_seed

In [3]:
def read_json(path):
    return json.loads(path.read_text(encoding="utf-8"))

MLLM_ROOT = Path("/content/drive/MyDrive/MLLM")
C_ROOT = MLLM_ROOT / "mordern_llm"
D_ROOT = MLLM_ROOT / "section_d"
DEVICE = torch.device("cuda:0")

E_ROOT = MLLM_ROOT / "section_e"
C_DATA = C_ROOT / "data/sroie"
PROCESSOR_DIR = C_ROOT / "outputs/qlora_run1/best_adapter"

PILOT = D_ROOT / "outputs/sft_pilot_v1"
RECIPE_PATH = PILOT / "recipe.json"
D_LOCK_PATH = D_ROOT / "data/heldout_manifest.json"
ADAPTER = PILOT / "llm_lora/best_adapter"

manifest = read_json(C_DATA / "manifest.json")
recipe = read_json(RECIPE_PATH)
CHECKPOINT_STEP = read_json(PILOT / "llm_lora/best_validation.json")["step"]

task = recipe["task"]
FIELDS = tuple(task["fields"])
DOC_PROMPT = task["prompt"]
EVAL_BUDGET = recipe["image_kwargs"]

DTYPE = {"torch.float16": torch.float16, "torch.bfloat16": torch.bfloat16}[recipe["dtype"]]

if DTYPE == torch.bfloat16 and torch.cuda.get_device_capability(DEVICE)[0] < 8:
    raise RuntimeError("Use a GPU with native BF16 support to match the saved recipe.")

processor = AutoProcessor.from_pretrained(PROCESSOR_DIR, use_fast=False)
processor.tokenizer.padding_side = "right"
E_ROOT.mkdir(parents=True, exist_ok=True)

print("Section C:", C_ROOT)
print("Section D:", D_ROOT)
print("Processor:", PROCESSOR_DIR)
print("LoRA checkpoint step:", CHECKPOINT_STEP)
print("Dtype:", DTYPE)

Section C: /content/drive/MyDrive/MLLM/mordern_llm
Section D: /content/drive/MyDrive/MLLM/section_d
Processor: /content/drive/MyDrive/MLLM/mordern_llm/outputs/qlora_run1/best_adapter
LoRA checkpoint step: 8
Dtype: torch.float16


In [ ]:
def file_sha(path):
    with path.open("rb") as f:
        return hashlib.file_digest(f, "sha256").hexdigest()

def write_json(path, data):
    path.write_text(json.dumps(data, ensure_ascii=False, indent=2), encoding="utf-8")

def normalize_ocr(text):
    return " ".join(unicodedata.normalize("NFC", text).casefold().split())

def lock_json(path, data):
    if path.exists():
        assert read_json(path) == data, f"Existing lock differs: {path.name}"
    else:
        write_json(path, data)

def normal(field, value):
    if not isinstance(value, str) or not value.strip():
        return None

    text = " ".join(unicodedata.normalize("NFKC", value).casefold().split())
    if field != "total":
        return text
    
    text = re.sub(r"^(?:rm|\$)\s*", "", text)
    if not re.fullmatch(
        r"[+-]?(?:\d+|\d{1,3}(?:,\d{3})+)(?:\.\d+)?", text
    ):
        return None

    amount = Decimal(text.replace(",", ""))
    return format(amount.normalize(), "f")

In [5]:
def score_doc(answer, reference):
    raw_valid = True
    used_cleanup = False

    try:
        prediction = json.loads(answer)
    except json.JSONDecodeError:
        raw_valid = False
        prediction = None

        fence = re.fullmatch(r"```(?:json)?\s*(.*?)\s*```", answer.strip(), flags=re.IGNORECASE | re.DOTALL)

        if fence:
            used_cleanup = True
            try:
                prediction = json.loads(fence.group(1))
            except json.JSONDecodeError:
                pass

    schema_valid = (
        isinstance(prediction, dict)
        and set(prediction) == set(FIELDS)
        and all(value is None or isinstance(value, str) for value in prediction.values())
    )

    flags = {}

    for field in FIELDS:
        correct = False

        if schema_valid:
            predicted = prediction[field]
            expected = reference[field]

            if predicted is None or expected is None:
                correct = predicted is None and expected is None
            else:
                normalized = normal(field, predicted)
                correct = (
                    normalized is not None
                    and normalized == normal(field, expected)
                )

        flags[f"{field}_correct"] = correct

    return {
        "prediction": prediction,
        "raw_json_valid": raw_valid,
        "schema_valid": schema_valid,
        "used_fence_cleanup": used_cleanup,
        **flags,
        "all_correct": all(flags.values())
    }

In [6]:
CAPS = {"vqa": 16, "receipt": task["max_new_tokens"]}
adapter_hashes = [
    file_sha(ADAPTER / "adapter_config.json"),
    file_sha(ADAPTER / "adapter_model.safetensors")
]

protocol = {
    "base_id": recipe["model_id"],
    "base_revision": recipe["revision"],
    "adapter_path": str(ADAPTER),
    "adapter_hashes": adapter_hashes,
    "adapter_checkpoint_step": CHECKPOINT_STEP,
    "dtype": str(DTYPE),
    "quantization": None,
    "attention_implementation": "sdpa",
    "processor": str(PROCESSOR_DIR),
    "processor_use_fast": False,
    "image_kwargs": EVAL_BUDGET,
    "generation": {
        "max_new_tokens_by_task": CAPS,
        "do_sample": False,
        "use_cache": True,
        "use_model_defaults": False,
        "eos_token_id": processor.tokenizer.eos_token_id,
        "pad_token_id": processor.tokenizer.pad_token_id
    },
    "seed" : 42,
    "gpu" : torch.cuda.get_device_name(DEVICE),
    "versions": {name: version(name) for name in ("torch", "transformers", "peft", "accelerate")},
    "dev_manifest_sha256": file_sha(E_ROOT / "dev_samples_v1.json"),
    "scoring_rules": ["closed_choice_v1", "receipt_c_v1"],
    "failed_requests_in_denominator": True
}

RUN_ID = hashlib.sha256(json.dumps(protocol, sort_keys=True).encode("utf-8")).hexdigest()[:12]

E_OUT = E_ROOT / "outputs" / RUN_ID
E_OUT.mkdir(parents=True, exist_ok=True)
lock_json(E_OUT / "protocol.json", protocol)

print("Run folder:", E_OUT)

Run folder: /content/drive/MyDrive/MLLM/section_e/outputs/ba2d7237c1e2


In [7]:
def load_eval_model(setting):
    if setting not in ("base", "llm_lora"):
        raise ValueError(f"Unknown evaluation setting: {setting}")

    net = Qwen3VLForConditionalGeneration.from_pretrained(
        recipe["model_id"],
        revision=recipe["revision"],
        dtype=DTYPE,
        device_map={"": 0},
        attn_implementation="sdpa"
    )
    if setting == "llm_lora":
        net = PeftModel.from_pretrained(net, ADAPTER, is_trainable=False)

    return net.eval()


@torch.no_grad()
def predict_sample(net, sample):
    messages = [{
        "role": "user",
        "content": [
            {"type": "image"},
            {"type": "text", "text": sample["question"]}
        ],
    }]
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

    with Image.open(sample["image_path"]) as source:
        image = source.convert("RGB")

    batch = processor(
        text=[text], images=[image], images_kwargs=EVAL_BUDGET,
        add_special_tokens=False, return_tensors="pt"
    ).to(DEVICE)

    config = GenerationConfig(
        do_sample=False,
        use_cache=True,
        max_new_tokens=CAPS[sample["task"]],
        eos_token_id=processor.tokenizer.eos_token_id,
        pad_token_id=processor.tokenizer.pad_token_id
    )

    with torch.autocast("cuda", dtype=DTYPE):
        output = net.generate(**batch, generation_config=config, use_model_defaults=False)

    new_ids = output[0, batch["input_ids"].shape[1]:]
    return processor.tokenizer.decode(new_ids, skip_special_tokens=True)

**OCR, Grounding and Receipt Extraction**

In [ ]:
D3_OUT = E_OUT / "day3_task_quality"
ASSETS = D3_OUT / "images"
ASSETS.mkdir(parents=True, exist_ok=True)
CAPS = {**CAPS, "ocr": 128, "grounding": 96}

def make_sample(task, sample_id, image, question, expected):
    path = ASSETS / f"{sample_id}.png"
    image.save(path)
    return {
        "sample_id": sample_id,
        "task": task,
        "image_path": str(path),
        "image_sha256": file_sha(path),
        "size": list(image.size),
        "question": question,
        "expected": expected,
        "split": "regression"
    }

def fixed_samples(name, build):
    path = D3_OUT / f"{name}_samples.json"
    if not path.exists():
        write_json(path, build())
    return read_json(path)

In [ ]:
def build_ocr_samples():
    records = read_json(D_LOCK_PATH)["test"][:8]
    paths = {}
    for r in records:
        rid = r["id"]
        path = D_ROOT / (r.get("image_path") or f"data/heldout_images/{rid}.jpg")
        if not path.is_file():
            path = D_ROOT / f"data/heldout_images/{rid}.jpg"
        assert path.is_file(), f"Receipt image missing: {path}"
        paths[rid] = path

    url = (
        "https://huggingface.co/datasets/jsdnrs/ICDAR2019-SROIE/resolve/"
        f"{manifest['revision']}/data/test-00000-of-00001.parquet"
    )
    annotations = load_dataset(
        "parquet", data_files={"test": url}, split="test", streaming=True,
        columns=["key", "image_size", "words", "bboxes"]
    )

    samples = []
    for row in annotations:
        rid = Path(row["key"]).stem
        if rid not in paths:
            continue

        with Image.open(paths.pop(rid)) as source:
            image = source.convert("RGB")
        w, h = image.size

        lines = []
        for i, (text, box) in enumerate(zip(row["words"], row["bboxes"])):
            x1, y1, x2, y2 = box
            if len(text.strip()) >= 4 and 0 <= x1 < x2 <= w and 0 <= y1 < y2 <= h:
                lines.append((i, text, box))

        for j in (0, len(lines) // 2, len(lines) - 1):
            i, text, box = lines[j]
            samples.append(make_sample(
                "ocr", f"ocr_{rid}_{i}", image.crop(box),
                "Transcribe the text in this crop. Output only the text.", text,
            ))
        if not paths:
            break
    return samples

OCR = fixed_samples("ocr", build_ocr_samples)
print("OCR crops:", len(OCR))

In [ ]:
def build_grounding_samples():
    ids = """
    1537681 710706 480908 469475 1746473 1741008 190137 514243 2159030 507653
    442199 50194 49369 473903 466635 2162742 513538 1743532 485679 477161
    """.split()
    dataset = load_dataset(
        "lmms-lab-encoder/RefCOCO", split="val", streaming=True,
        revision="a5dff0b3194715fda69b2d6a0d2aaafb41eaa407"
    )
    rows = list(dataset.take(20))

    samples = []
    for r in rows:
        x, y, width, height = r["bbox"]
        phrase = r["answer"][0]
        question = (
            f'Locate "{phrase}". Return one JSON object with key "bbox_2d". '
            "Use [x1, y1, x2, y2] normalized to 0–1000."
        )
        samples.append(make_sample(
            "grounding", f"ground_{r['question_id']}", r["image"].convert("RGB"),
            question, [x, y, x + width, y + height]
        ))
    return samples

GROUND = fixed_samples("grounding", build_grounding_samples)
SAMPLES = OCR + GROUND
print(f"{len(OCR)} OCR + {len(GROUND)} grounding = {len(SAMPLES)} per model")

In [ ]:
def edit_distance(a, b):
    previous = list(range(len(b) + 1))
    for i, ca in enumerate(a, 1):
        current = [i]
        for j, cb in enumerate(b, 1):
            current.append(min(
                current[-1] + 1,             
                previous[j] + 1,             
                previous[j - 1] + (ca != cb)
            ))
        previous = current
    return previous[-1]

def parse_box(raw, w, h):
    fence = re.fullmatch(r"```(?:json)?\s*(.*?)\s*```", raw.strip(), re.I | re.S)
    try:
        obj = json.loads(fence[1] if fence else raw)
    except json.JSONDecodeError:
        return None

    if isinstance(obj, list) and len(obj) == 1:
        obj = obj[0]

    b = obj.get("bbox_2d") if isinstance(obj, dict) else None
    if not (isinstance(b, list) and len(b) == 4
            and all(type(v) in (int, float) and 0 <= v <= 1000 for v in b)
            and b[0] < b[2] and b[1] < b[3]):
        return None

    return [v * scale / 1000 for v, scale in zip(b, (w, h, w, h))]

def iou(a, b):
    overlap_w = max(0, min(a[2], b[2]) - max(a[0], b[0]))
    overlap_h = max(0, min(a[3], b[3]) - max(a[1], b[1]))
    intersection = overlap_w * overlap_h
    area_a = (a[2] - a[0]) * (a[3] - a[1])
    area_b = (b[2] - b[0]) * (b[3] - b[1])
    return intersection / (area_a + area_b - intersection)

def score_day3(sample, raw):
    if sample["task"] == "ocr":
        parsed = normalize_ocr(raw)
        ref = normalize_ocr(sample["expected"])
        valid, correct = bool(parsed), parsed == ref
        metrics = {"edits": edit_distance(ref, parsed), "ref_chars": len(ref)}
    else:
        parsed = parse_box(raw, *sample["size"])
        valid = parsed is not None
        overlap = iou(parsed, sample["expected"]) if valid else 0.0
        correct = overlap >= 0.5
        metrics = {"iou": overlap, "invalid_box": not valid}

    return {
        "parsed_answer": parsed,
        "score": int(correct),
        "status": "correct" if correct else "wrong_answer" if valid else "invalid_output",
        **metrics
    }

print("Scorer checks passed")